# From signal to net P&L, on synthetic data

**Evidence class: synthetic illustration. Nothing here is a finding about markets.**

This notebook walks one daily cross-sectional signal through the four steps every family in this repository goes
through: signal, portfolio, costs, and the E5 report blocks. The data are simulated with a small planted
short-term reversal, so the gross line is positive by construction. The point is to show how much of it survives
turnover and costs, and how the deflated Sharpe ratio and PBO read the result.

It runs top to bottom in well under a minute. It needs numpy, pandas and matplotlib, and the repository's
`python/src` on the path (the first cell adds it).

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "python" / "src" / "iap").is_dir())
sys.path.insert(0, str(ROOT / "python" / "src"))

import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402
from iap.reports import (  # noqa: E402
    SessionRow,
    block_d_risk_adjusted,
    block_f_overfitting,
    session_series,
)

rng = np.random.default_rng(20261027)

## 1. Synthetic returns with a planted reversal

`N` names over `T` sessions. Each name's return is a market factor times its beta, plus an idiosyncratic part that
gives back a fraction `PLANTED` of its previous five sessions' idiosyncratic move. All returns are in bps.

In [ ]:
N, T = 200, 750
PLANTED = 0.01  # share of the trailing five-session residual move that reverses next session
SIGMA_IDIO = 150.0  # bps per session
beta = rng.normal(1.0, 0.3, N)
mkt = rng.normal(2.0, 90.0, T)
eps = rng.normal(0.0, SIGMA_IDIO, (T, N))
resid = np.zeros((T, N))
for t in range(T):
    resid[t] = eps[t] - PLANTED * resid[max(t - 5, 0) : t].sum(axis=0)
ret = mkt[:, None] * beta[None, :] + resid
ret.shape

## 2. Signal

The same shape as D1: minus the five-session residual return, scaled by its trailing dispersion, capped at 4.
Only sessions before `t` enter the signal for session `t`. Residuals use the true beta here; the repository's
loop estimates a trailing 60-session beta instead.

In [ ]:
def zscore_signal(resid, lookback=5, vol_window=60):
    T, N = resid.shape
    z = np.full((T, N), np.nan)
    for t in range(vol_window, T):
        s = resid[t - lookback : t].sum(axis=0)  # sessions t-lookback..t-1: known before t
        sd = resid[t - vol_window : t].std(axis=0, ddof=1) * np.sqrt(lookback)
        z[t] = np.clip(-s / sd, -4, 4)
    return z


signal = zscore_signal(resid)
live = ~np.isnan(signal).any(axis=1)
ic = [np.corrcoef(signal[t], resid[t])[0, 1] for t in np.where(live)[0]]
print(f"mean daily IC {np.mean(ic):.4f} over {len(ic)} sessions")

## 3. Portfolio

Long the top 20 % and short the bottom 20 % by signal, equal weight, dollar neutral, gross exposure 1. A
dollar-neutral book still carries some market beta, because the market enters each name through its own beta; it
is printed, not hedged, to keep the example short.

In [ ]:
def tails_book(z, q=0.2):
    w = np.zeros_like(z)
    k = int(q * z.shape[1])
    for t in range(z.shape[0]):
        if np.isnan(z[t]).any():
            continue
        order = np.argsort(z[t])
        w[t, order[-k:]] = 0.5 / k
        w[t, order[:k]] = -0.5 / k
    return w


def one_way_turnover(w):
    return np.abs(np.diff(w, axis=0, prepend=0.0)).sum(axis=1)


w = tails_book(signal)
turnover = one_way_turnover(w)
gross_bps = (w * ret).sum(axis=1)
print(f"mean one-way turnover {turnover[live].mean():.2f} of capital per session")
print(f"mean book beta {np.mean([(w[t] * beta).sum() for t in np.where(live)[0]]):+.3f}")

## 4. Costs

A flat one-way cost per unit of capital traded, `COST_BPS`, standing in for half the spread plus fees plus impact.
Net is gross minus turnover times that cost. This is the decomposition the one-pager uses:

net = gross signal - cost drag, and cost drag = turnover x unit cost.

In [ ]:
COST_BPS = 5.0
cost_bps = turnover * COST_BPS
net_bps = gross_bps - cost_bps
g, c, n = gross_bps[live], cost_bps[live], net_bps[live]
print(f"gross {g.mean():+.2f} bps/session, cost drag {c.mean():.2f}, net {n.mean():+.2f}")
print(f"break-even one-way cost {g.mean() / turnover[live].mean():.2f} bps")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(np.cumsum(g), label="gross")
ax.plot(np.cumsum(n), label=f"net of {COST_BPS:.0f} bps one-way")
ax.axhline(0, color="grey", lw=0.8)
ax.set_xlabel("session")
ax.set_ylabel("cumulative bps of capital")
ax.set_title("Synthetic reversal: gross vs net")
ax.legend()
plt.show()

## 5. E5 blocks D and F

The repository's report builders take a session series on registered capital `C`. Block D gives the per-session
Sharpe with a block-bootstrap interval. Block F deflates it for the number of trials and, given a sweep, estimates
PBO by combinatorially symmetric cross-validation (CSCV).

In [ ]:
C = 1_000_000.0


def series_of(net, gross):
    rows = [
        SessionRow(f"s{i}", "traded", x * C / 1e4, y * C / 1e4)
        for i, (x, y) in enumerate(zip(net, gross, strict=True))
    ]
    return session_series("demo", "1d", rows, C)


d = block_d_risk_adjusted(series_of(n, g), bootstrap=200).rows[0]
print(
    f"net per-session Sharpe {d['net_session_sharpe']:.3f}, 95 % block bootstrap {d['sharpe_block_bootstrap_95']}"
)

A sweep: the same rule at eight lookbacks, 1 to 20 sessions. The best in-sample lookback is the one an
unwary researcher would report. The deflated Sharpe ratio charges for the eight tries; PBO asks how often the
in-sample winner lands in the bottom half out of sample.

In [ ]:
lookbacks = [1, 2, 3, 5, 8, 10, 15, 20]
sweep = []
for lb in lookbacks:
    wl = tails_book(zscore_signal(resid, lookback=lb))
    sweep.append(((wl * ret).sum(axis=1) - one_way_turnover(wl) * COST_BPS)[live])
M = np.column_stack(sweep)
best = int(np.argmax(M.mean(axis=0) / M.std(axis=0, ddof=1)))
f = block_f_overfitting(series_of(M[:, best], M[:, best]), n_trials=len(lookbacks), candidates=M)
r0, pbo = f.rows[0], f.rows[1]["probability_of_backtest_overfitting"]
print(
    f"best lookback {lookbacks[best]}: Sharpe {r0['session_sharpe']:.3f}, SR0 for {r0['n_trials']} trials {r0['expected_max_sharpe_sr0']:.3f}"
)
print(
    f"deflated Sharpe ratio {r0['deflated_sharpe_ratio']:.3f}; PBO {pbo['pbo']:.2f} over {pbo['n_splits']} CSCV splits"
)

## What to take from it

* The gross line is the signal; the gap to the net line is turnover times unit cost. A reversal signal replaces
  much of its book every session, so a modest unit cost removes a large share of a small gross edge.
* The break-even cost from step 4 is the first number to compare with a realistic cost estimate.
* The deflated Sharpe ratio and PBO price the search. With eight lookbacks the hurdle SR0 is above zero, so a
  net Sharpe near it is not evidence of anything.

The recorded D1 and D3 results on real data are discussed in `docs/QUANT_ONE_PAGER.md`.